# SciFact HyDE Dense Retrieval

Retrieval-only HyDE experiment on SciFact (query -> hypothetical document -> embedding -> retrieval).

In [ ]:
!pip install openai

In [1]:
import os

# Must run before importing numpy/torch/sentence-transformers
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

try:
    import torch
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception:
    pass

print('OpenMP/BLAS thread guards enabled.')


OpenMP/BLAS thread guards enabled.


In [2]:
import os
from getpass import getpass

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key (OPENAI_API_KEY): ').strip()

print('OPENAI_API_KEY is set:', bool(os.getenv('OPENAI_API_KEY')))


OPENAI_API_KEY is set: True


In [3]:
import importlib, inspect
import load_data
load_data = importlib.reload(load_data)
print('load_data module file:', load_data.__file__)
print('load_scifact_data starts at line:', inspect.getsourcelines(load_data.load_scifact_data)[1])
src = inspect.getsource(load_data.load_scifact_data)
print('contains _load_raw_scifact?', '_load_raw_scifact' in src)


/Users/winstondong/miniforge3/envs/adnlp_hyde_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


load_data module file: /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3&4/baseline_reproduction/ANLP-HW34/hydeOnSciFact/load_data.py
load_scifact_data starts at line: 130
contains _load_raw_scifact? False


In [4]:
from pathlib import Path
import json
from tqdm import tqdm

from config import DATASET_SPLIT, PREFER_BEIR, MAX_QUERIES, TOP_KS, EMBED_MODEL, LLM_MODEL
from load_data import load_scifact_data
from embed import Embedder
from retrieve import build_faiss_index, retrieve_top_k
from evaluate import evaluate_run
from generate_hyde import HyDEGenerator


In [5]:
# Inspect SciFact corpus document length distribution
import numpy as np

# Reuse already loaded corpus if available; otherwise load quickly.
if 'corpus' not in globals():
    corpus, _, _, data_source_for_stats = load_scifact_data(
        split=DATASET_SPLIT,
        prefer_beir=PREFER_BEIR,
        max_queries=None,
    )
else:
    data_source_for_stats = 'already-loaded corpus in memory'

doc_items = list(corpus.items())
doc_texts = [text for _, text in doc_items]

char_lens = np.array([len(t) for t in doc_texts], dtype=np.int32)
word_lens = np.array([len(t.split()) for t in doc_texts], dtype=np.int32)

def _fmt_percentiles(arr):
    ps = [0, 25, 50, 75, 90, 95, 99, 100]
    return ', '.join([f'p{p}={np.percentile(arr, p):.1f}' for p in ps])

print(f'Data source: {data_source_for_stats}')
print(f'Doc count : {len(doc_texts)}')
print('\n[Chars]')
print(_fmt_percentiles(char_lens), f', mean={char_lens.mean():.1f}')
print('\n[Words: whitespace split]')
print(_fmt_percentiles(word_lens), f', mean={word_lens.mean():.1f}')

bins = [0, 50, 100, 150, 200, 250, 300, 400, 500, 1000, 10000]
hist = np.histogram(word_lens, bins=bins)[0]
print('\n[Word-length histogram]')
for i, c in enumerate(hist):
    print(f'[{bins[i]:>4}, {bins[i+1]:>5}): {int(c)}')

longest_idx = np.argsort(word_lens)[-5:][::-1]
print('\n[Top-5 longest docs]')
for rank, idx in enumerate(longest_idx, start=1):
    doc_id, text = doc_items[int(idx)]
    preview = text.replace('\n', ' ')[:180]
    print(f'#{rank} doc_id={doc_id} words={int(word_lens[idx])} chars={int(char_lens[idx])}')
    print(f'   preview: {preview}')


Generating queries split: 100%|██████████| 1109/1109 [00:00<00:00, 234947.12 examples/s]


Data source: mteb/scifact (corpus:corpus/corpus, queries:queries/queries, qrels:default/test)
Doc count : 5183

[Chars]
p0=237.0, p25=1140.0, p50=1442.0, p75=1827.5, p90=2140.0, p95=2452.9, p99=3163.4, p100=10143.0 , mean=1515.4

[Words: whitespace split]
p0=35.0, p25=160.0, p50=206.0, p75=262.0, p90=311.0, p95=358.0, p99=475.0, p100=1543.0 , mean=216.6

[Word-length histogram]
[   0,    50): 3
[  50,   100): 157
[ 100,   150): 765
[ 150,   200): 1489
[ 200,   250): 1193
[ 250,   300): 958
[ 300,   400): 481
[ 400,   500): 99
[ 500,  1000): 33
[1000, 10000): 5

[Top-5 longest docs]
#1 doc_id=10749308 words=1543 chars=10143
   preview: Title: Placebo-Controlled Trials and Active-Control Trials in the Evaluation of New Treatments. Part 1: Ethical and Scientific Issues Passage: Placebo-controlled trials are used ex
#2 doc_id=26067999 words=1524 chars=10104
   preview: Title: Screening for Lung Cancer: U.S. Preventive Services Task Force Recommendation Statement Passage: The U.S. Preventiv

In [ ]:
# Config for this notebook run
SPLIT = DATASET_SPLIT
PREFER_BEIR_LOCAL = PREFER_BEIR
MAX_QUERIES_LOCAL = MAX_QUERIES
TOP_KS_LOCAL = TOP_KS
MAX_K = max(TOP_KS_LOCAL)
OUTPUT_DIR = Path('results/hyde')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

corpus, queries, qrels, data_source = load_scifact_data(
    split=SPLIT,
    prefer_beir=PREFER_BEIR_LOCAL,
    max_queries=MAX_QUERIES_LOCAL,
)

query_ids = list(queries.keys())
corpus_doc_ids = list(corpus.keys())
corpus_texts = [corpus[did] for did in corpus_doc_ids]

print(f'Loaded dataset source: {data_source}')
print(f'Corpus size: {len(corpus_texts)} | Queries: {len(query_ids)}')

embedder = Embedder(model_name=EMBED_MODEL)
corpus_vectors = embedder.encode(corpus_texts)
index = build_faiss_index(corpus_vectors)

LLM_MODEL_OVERRIDE = 'gpt-4o-mini'
PROVIDER_OVERRIDE = 'openai'
hyde_generator = HyDEGenerator(model_name=LLM_MODEL_OVERRIDE, provider=PROVIDER_OVERRIDE)
print('HyDE model selected:', hyde_generator.model_name, '| provider:', PROVIDER_OVERRIDE)
retrieval_texts = []
hyde_docs = {}
for qid in tqdm(query_ids, desc='Generate-HyDE'):
    hyde_doc = hyde_generator.generate(queries[qid])
    retrieval_texts.append(hyde_doc)
    hyde_docs[qid] = hyde_doc

query_vectors = embedder.encode(retrieval_texts)
_, all_indices = retrieve_top_k(index, query_vectors, MAX_K)

per_query_retrieved = {}
per_query_rows = []
for i, qid in enumerate(query_ids):
    retrieved_doc_ids = [corpus_doc_ids[j] for j in all_indices[i].tolist()]
    per_query_retrieved[qid] = retrieved_doc_ids

    row = {
        'mode': 'hyde',
        'query_id': qid,
        'query': queries[qid],
        'hyde_document': hyde_docs[qid],
        'gold_doc_ids': sorted(qrels[qid]),
        'retrieved_doc_ids': retrieved_doc_ids,
        'hit@1': int(any(d in qrels[qid] for d in retrieved_doc_ids[:1])),
        'hit@5': int(any(d in qrels[qid] for d in retrieved_doc_ids[:5])),
        'hit@10': int(any(d in qrels[qid] for d in retrieved_doc_ids[:10])),
    }
    per_query_rows.append(row)

metrics = evaluate_run(per_query_retrieved, qrels, TOP_KS_LOCAL)

print('\n=== HyDE Summary ===')
for k in ['Recall@1', 'Recall@5', 'Recall@10', 'MRR@10', 'nDCG@10']:
    if k in metrics:
        print(f'{k:<10}: {metrics[k]:.4f}')

metrics_payload = {
    'config': {
        'mode': 'hyde',
        'split': SPLIT,
        'prefer_beir': PREFER_BEIR_LOCAL,
        'data_source': data_source,
        'embed_model': EMBED_MODEL,
        'llm_model': LLM_MODEL,
        'top_ks': TOP_KS_LOCAL,
        'max_queries': MAX_QUERIES_LOCAL,
    },
    'hyde': metrics,
}

metrics_path = OUTPUT_DIR / 'metrics.json'
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)

per_query_path = OUTPUT_DIR / 'per_query_results.jsonl'
with per_query_path.open('w', encoding='utf-8') as f:
    for row in per_query_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'\nSaved metrics: {metrics_path}')
print(f'Saved per-query results: {per_query_path}')


In [ ]:
# Re-run retrieval + eval + save with TOP_KS=[1,10,100]
# Loads previously saved HyDE documents from disk — no API calls needed.

from pathlib import Path
import json
from config import DATASET_SPLIT, PREFER_BEIR, MAX_QUERIES, EMBED_MODEL, LLM_MODEL
from load_data import load_scifact_data
from embed import Embedder
from retrieve import build_faiss_index, retrieve_top_k
from evaluate import evaluate_run

TOP_KS_NEW = [1, 10, 100]
MAX_K_NEW  = max(TOP_KS_NEW)
OUTPUT_DIR = Path('results/hyde')

# 1. Load corpus / queries / qrels (fast, no model needed yet)
corpus, queries, qrels, data_source = load_scifact_data(
    split=DATASET_SPLIT, prefer_beir=PREFER_BEIR, max_queries=MAX_QUERIES
)
query_ids      = list(queries.keys())
corpus_doc_ids = list(corpus.keys())
corpus_texts   = [corpus[did] for did in corpus_doc_ids]

# 2. Load saved HyDE documents (no API call)
saved_path = OUTPUT_DIR / 'per_query_results.jsonl'
hyde_docs = {}
with saved_path.open() as f:
    for line in f:
        row = json.loads(line)
        hyde_docs[row['query_id']] = row['hyde_document']
print(f'Loaded {len(hyde_docs)} HyDE docs from {saved_path}')

# 3. Rebuild embedder + FAISS index
embedder       = Embedder(model_name=EMBED_MODEL)
corpus_vectors = embedder.encode(corpus_texts)
index          = build_faiss_index(corpus_vectors)

# 4. Encode HyDE docs and retrieve top-100
retrieval_texts = [hyde_docs[qid] for qid in query_ids]
query_vectors   = embedder.encode(retrieval_texts)
_, all_indices  = retrieve_top_k(index, query_vectors, MAX_K_NEW)

per_query_retrieved = {}
per_query_rows      = []
for i, qid in enumerate(query_ids):
    retrieved_doc_ids = [corpus_doc_ids[j] for j in all_indices[i].tolist()]
    per_query_retrieved[qid] = retrieved_doc_ids
    gold = qrels[qid]
    per_query_rows.append({
        'mode': 'hyde',
        'query_id': qid,
        'query': queries[qid],
        'hyde_document': hyde_docs[qid],
        'gold_doc_ids': sorted(gold),
        'retrieved_doc_ids': retrieved_doc_ids,
        'hit@1':   int(any(d in gold for d in retrieved_doc_ids[:1])),
        'hit@10':  int(any(d in gold for d in retrieved_doc_ids[:10])),
        'hit@100': int(any(d in gold for d in retrieved_doc_ids[:100])),
    })

# 5. Evaluate
metrics = evaluate_run(per_query_retrieved, qrels, TOP_KS_NEW)
print('\n=== HyDE (TOP_KS=[1,10,100]) ===')
for k in ['Recall@1', 'Recall@10', 'Recall@100', 'MRR@10', 'nDCG@10']:
    if k in metrics:
        print(f'{k:<12}: {metrics[k]:.4f}')

# 6. Save (overwrite)
metrics_payload = {
    'config': {
        'mode': 'hyde', 'split': DATASET_SPLIT, 'prefer_beir': PREFER_BEIR,
        'data_source': data_source, 'embed_model': EMBED_MODEL,
        'llm_model': LLM_MODEL, 'top_ks': TOP_KS_NEW, 'max_queries': MAX_QUERIES,
    },
    'hyde': metrics,
}
with (OUTPUT_DIR / 'metrics.json').open('w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)
with (OUTPUT_DIR / 'per_query_results.jsonl').open('w', encoding='utf-8') as f:
    for row in per_query_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'\nSaved to {OUTPUT_DIR}')

Loaded 300 HyDE docs from results/hyde/per_query_results.jsonl

=== HyDE (TOP_KS=[1,10,100]) ===
Recall@1    : 0.5683
Recall@10   : 0.8824
Recall@100  : 0.9667
MRR@10      : 0.6901
nDCG@10     : 0.7338

Saved to results/hyde
